In [0]:
%run ../04_Ukey_Match/ukey_match_regular_common_business

In [0]:
def check_empty_ukey(new_ukey_df):
    empty_ukey_count = new_ukey_df.filter(F.coalesce(F.col("new_consumermdmkey"), F.lit("")) == "").count()
    if empty_ukey_count > 0:
        raise ValueError("new_consumermdmkey must be non empty")


In [0]:
def update_ukey_to_master_table(task_id):

    new_ukey_df = (spark.table(f"{get_env_config('silver_consumer_cleansed_database')}.t_clean_ukey_group")
        .filter(F.col("task_id") == task_id) 
        .select(
            F.col("task_id"), 
            F.col("mrkt_code"), 
            F.col("master_scon_id"), 
            F.col("new_consumermdmkey"), 
            F.col("match_type")   
        )
        .distinct())

    check_empty_ukey(new_ukey_df)

    # 更新 master recode 中ukey 
    base_table = DeltaTable.forName(spark, f"{get_env_config('golden_consumer_master_database')}.t_master_consumer")

    # merge
    # regular consumer 和 rebind consumer 进行不同主键的merger
    (base_table.alias("b")
        .merge(new_ukey_df.alias("u"), 
            f"""
                b.scon_mrkt_code = u.mrkt_code and b.scon_id = u.master_scon_id
            """
        )
        .whenMatchedUpdate(
            set = {
                "consumermdmkey": F.col("u.new_consumermdmkey"),
                "task_id": F.col("u.task_id"),
            }
        )
        .execute())

In [0]:
task_id = dbutils.widgets.get("task_id")
print(f"task_id: {task_id}")

update_ukey_to_master_table(task_id)